In [ ]:
# Load in Libraries
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor

# Load Used Car Dataset
df = pd.read_csv('~/Downloads/used_cars.csv')

# Preprocessing
# Clean Price, Title, and Fuel Type Columns
fuel_allowed = ["Gasoline", "Hybrid", "Diesel"]
df['price'] = df['price'].astype(str).str.replace(',', '').str.replace('$', '')
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df["clean_title"] = df["clean_title"].eq("Yes").astype(int)
df["fuel_type"] = df["fuel_type"].str.strip()
df["fuel_type"] = df["fuel_type"].replace("Plug-In Hybrid", "Hybrid")
df["fuel_type"] = df["fuel_type"].where(df["fuel_type"].isin(fuel_allowed), "Other")

# Cleaning Interior/Exterior Color Columns
def simplify_color(color):
    if pd.isna(color):
        return "Other"
    color = color.lower()
    if "black" in color:                              return "Black"
    elif "white" in color:                            return "White"
    elif "blue" in color or "blu" in color:           return "Blue"
    elif "red" in color or "rosso" in color:          return "Red"
    elif "gray" in color or "grey" in color or "graphite" in color: return "Gray"
    elif "silver" in color:                           return "Silver"
    elif "green" in color or "verde" in color:        return "Green"
    elif "brown" in color:                            return "Brown"
    elif "beige" in color or "tan" in color:          return "Beige"
    elif "yellow" in color:                           return "Yellow"
    elif "orange" in color or "mango" in color:       return "Orange"
    elif "purple" in color or "plum" in color:        return "Purple"
    elif "gold" in color:                             return "Gold"
    elif "pink" in color:                             return "Pink"
    else:                                             return "Other"

df["ext_col"] = df["ext_col"].str.strip().str.lower().apply(simplify_color)
df["int_col"] = df["int_col"].str.strip().str.lower().apply(simplify_color)

# Cleaning Accident Column
def simplify_acc(accident):
    if pd.isna(accident):
        return 0
    return 1 if "at least 1 accident or damage reported" in accident.lower() else 0

df["accident"] = df["accident"].apply(simplify_acc)

# Cleaning Transmission Column
def simplify_transmission(trans):
    if pd.isna(trans):
        return "Other"
    trans = str(trans).lower().strip()
    if "manual" in trans or "m/t" in trans or "mt" in trans:
        return "Manual"
    elif "cvt" in trans or "variable" in trans:
        return "CVT"
    elif any(x in trans for x in ["automatic", "a/t", "dual", "dct", "tronic", "steptronic", "pdk", "auto"]):
        return "Automatic"
    else:
        return "Other"

df["transmission"] = df["transmission"].apply(simplify_transmission)

# Cleaning Engine Column
df["engine"] = (df["engine"].astype(str).str.lower().str.strip()
                .str.replace(r'\d+\.?\d*\s*hp', '', regex=True))

df["is_electric"] = df["engine"].str.contains("electric", na=False).astype(int)
df["liters"] = pd.to_numeric(
    df["engine"].str.extract(r'(\d+\.\d+)\s*(?:l|liter)', expand=False), errors="coerce")
df["cylinders"] = pd.to_numeric(
    df["engine"].str.extract(r'\b(?:[viwh](\d+)|(\d+)\s*cylinder)\b')
    .bfill(axis=1).iloc[:, 0], errors="coerce")
df = df.drop(columns=['engine'])

# Cleaning Brand Column
def simplify_brand(brand):
    luxury  = {"BMW", "Mercedes-Benz", "Audi", "Lexus", "Porsche", "Jaguar",
                "Maserati", "Bentley", "Ferrari", "Lamborghini", "Rolls-Royce",
                "McLaren", "Aston", "Maybach", "Genesis", "Lucid", "Polestar"}
    premium = {"Acura", "INFINITI", "Volvo", "Cadillac", "Lincoln", "Tesla"}
    mass    = {"Ford", "Toyota", "Honda", "Chevrolet", "Hyundai", "Nissan",
                "Kia", "Subaru", "Mazda", "Volkswagen", "GMC", "RAM", "Jeep",
                "Dodge", "Chrysler", "Mitsubishi", "Buick", "FIAT", "Suzuki"}
    if brand in luxury:  return "Luxury"
    elif brand in premium: return "Premium"
    elif brand in mass:  return "Mass Market"
    else:                return "Other"

df["brand_type"] = df["brand"].apply(simplify_brand)
df = df.drop(columns=['brand'])

# Cleaning Mileage Column
df['milage'] = (df['milage'].astype(str)
                .str.replace(',', '', regex=False)
                .str.replace('mi.', '', regex=False)
                .str.strip())
df['milage'] = pd.to_numeric(df['milage'], errors='coerce')

# Model Encoding
le_model = LabelEncoder()
df['model_encoded'] = le_model.fit_transform(df['model'].astype(str))
df = df.drop(columns=['model'])

# Accounting for Missing Values
df = df.dropna(subset=['price'])

# Makes sure that electric cars are treated as NaN in the cylinders & liters column
df.loc[df['is_electric'] == 1, ['cylinders', 'liters']] = np.nan

# Remove Outliers
for col in ['price', 'milage']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[df[col].between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)]

# Price Log Transformation
df['log_price'] = np.log1p(df['price'])

# Label Encoding Categorical Columns
cat_cols = ['fuel_type', 'transmission', 'ext_col', 'int_col', 'brand_type']
le_cats = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_cats[col] = le

# Define the Model's Training Columns and Target
X = df.drop(columns=['price', 'log_price'])
y = df['log_price']

# Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
oof_preds = cross_val_predict(model, X, y, cv=kf)
oof_actuals = np.expm1(y)

print("5-Fold Cross Validation Results")
print(f"MAE:   ${mean_absolute_error(oof_actuals, np.expm1(oof_preds)):,.0f}")
print(f"R²:    {r2_score(oof_actuals, np.expm1(oof_preds)):.4f}")
print(f"MAPE:  {mean_absolute_percentage_error(oof_actuals, np.expm1(oof_preds)):.2%}")

# Refit on Full Data
model.fit(X, y)
joblib.dump(model, 'used_car_price_model.pkl')
joblib.dump(le_model,    'label_encoder_model.pkl')
joblib.dump(le_cats,     'label_encoders_cats.pkl')

# Final Prediction Function
def predict_price(model_year, milage, fuel_type, transmission, ext_col,
                  int_col, brand_type, accident, clean_title,
                  is_electric, liters, cylinders, model_name):
  
    # Encode categoricals
    def safe_encode(encoder, val):
        try:
            return encoder.transform([val])[0]
        except ValueError:
            return 0

    input_data = pd.DataFrame([{
        'model_year':    model_year,
        'milage':        milage,
        'fuel_type':     safe_encode(le_cats['fuel_type'],     fuel_type),
        'transmission':  safe_encode(le_cats['transmission'],  transmission),
        'ext_col':       safe_encode(le_cats['ext_col'],       ext_col),
        'int_col':       safe_encode(le_cats['int_col'],       int_col),
        'accident':      accident,
        'clean_title':   clean_title,
        'is_electric':   is_electric,
        'liters':        liters if not is_electric else np.nan,
        'cylinders':     cylinders if not is_electric else np.nan,
        'brand_type':    safe_encode(le_cats['brand_type'],    brand_type),
        'model_encoded': safe_encode(le_model,                 model_name)
    }])

    return np.expm1(model.predict(input_data)[0])


# Example of Price Prediction
price = predict_price(
    model_year=2008,
    milage=140000,
    fuel_type='Gasoline',
    transmission='Automatic',
    ext_col='Blue',
    int_col='Gray',
    brand_type='Mass Market',
    accident=0,
    clean_title=1,
    is_electric=0,
    liters=2.0,
    cylinders=4,
    model_name='Civic'
)
print(f"Example prediction: ${price:,.0f}")